# 02 · Data Cleaning

**Project:** Data Analyst – Mental Health (Canada) · **Pipeline step:** 2 of 10

> **Skeleton only.** This notebook currently just wires up the libraries and the
> `data/raw` → `data/processed` connections so the team can start cleaning tomorrow.
> The cleaning logic goes under section 4.


## 1 · Libraries

In [12]:
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
print("pandas", pd.__version__, "| numpy", np.__version__)


pandas 3.0.5 | numpy 2.5.2


## 2 · Paths (`data/raw` → `data/processed`)

In [13]:
def find_root(start: Path) -> Path:
    """Walk up until we find the folder that contains data/raw (works from repo root or /notebooks)."""
    for p in [start, *start.parents]:
        if (p / "data" / "raw").is_dir():
            return p
    raise FileNotFoundError("Could not find data/raw above " + str(start))

ROOT = find_root(Path.cwd())
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

print("root     :", ROOT)
print("raw      :", RAW)
print("processed:", PROCESSED)


root     : /home/codespace/createprojectfolder/Data-Analyst-Mental-Health-Project
raw      : /home/codespace/createprojectfolder/Data-Analyst-Mental-Health-Project/data/raw
processed: /home/codespace/createprojectfolder/Data-Analyst-Mental-Health-Project/data/processed


## 3 · Load the raw datasets

Same registry as `01_data_understanding.ipynb`. StatCan CSVs need `utf-8-sig` (BOM).
The CIHI Excel workbook is loaded separately (only the two hidden data sheets are usable).

In [14]:
DATASETS = {
    "perceived_mh_annual":        {"file": "StatCan 13-10-0972 – perceived mental health.csv",                                              "kind": "statcan_long"},
    "suicidal_thoughts":          {"file": "Catalogue Entry Mental health characteristics and suicidal thoughts.csv",                        "kind": "statcan_long"},
    "stress_coping":              {"file": "Catalogue Entry Mental health characteristics Ability to handle stress and sources of stress.csv","kind": "statcan_long"},
    "perceived_health_quarterly": {"file": "Catalogue Entry Mental health indicators.csv",                                                   "kind": "statcan_long"},
    "cchs_mh_disorders":          {"file": "Catalogue Entry Perceived health, by gender and province.csv",                                   "kind": "statcan_long"},
    "cihi_mh_services":           {"file": "health services for mental illness and alcoholdrug induced disorders.csv",                       "kind": "cihi_vizconfig"},
    "cihi_children_youth":        {"file": "care-children-youth-with-mental-disorders-data-tables-en.xlsx",                                  "kind": "excel_multitable"},
    "mhacs_2022_pumf":            {"file": "MHACS 2022 Public Use Microdata.csv",                                                            "kind": "microdata"},
}

def load_dataset(key: str) -> pd.DataFrame:
    spec = DATASETS[key]
    path = RAW / spec["file"]
    if spec["kind"] in ("statcan_long", "cihi_vizconfig"):
        return pd.read_csv(path, encoding="utf-8-sig", low_memory=False)
    if spec["kind"] == "microdata":
        return pd.read_csv(path, low_memory=False)
    raise ValueError(f"{key}: kind={spec['kind']} is loaded separately (see below)")

# CSV datasets -> raw[...]
raw = {k: load_dataset(k) for k, v in DATASETS.items() if v["kind"] != "excel_multitable"}

# CIHI workbook: the two machine-readable sheets (title row 0, headers row 1)
_xls = pd.ExcelFile(RAW / DATASETS["cihi_children_youth"]["file"])
raw_excel = {}
for _sheet in [s for s in _xls.sheet_names if s.endswith("_to hide")]:
    _t = _xls.parse(_sheet, header=None)
    _body = _t.iloc[2:].reset_index(drop=True)
    _body.columns = [str(h).replace("\n", " ").strip() for h in _t.iloc[1]]
    raw_excel[_sheet] = _body

for k, df in raw.items():
    print(f"{k:28} {df.shape}")
for k, df in raw_excel.items():
    print(f"{k:28} {df.shape}  (excel)")


perceived_mh_annual          (936, 18)
suicidal_thoughts            (8208, 18)
stress_coping                (27360, 18)
perceived_health_quarterly   (6318, 17)
cchs_mh_disorders            (160992, 18)
cihi_mh_services             (264, 14)
mhacs_2022_pumf              (9861, 602)
Table8DATA_to hide           (216, 13)  (excel)
Table13DATA_to hide          (216, 13)  (excel)


## 4 · Cleaning — TO BE COMPLETED BY THE TEAM

Write cleaned outputs to `PROCESSED / "..."`. Suggested tasks (see `docs/data_dictionary.md` §A):

- [ ] StatCan tables: melt to tidy long; pivot `Characteristics`/`Statistics` into `value` / `ci_low` / `ci_high` / `cv`
- [ ] Standardise `GEO` to one canonical province list; handle region rollups separately
- [ ] Parse `REF_DATE` (year / year-range / year-month) into a real date/period
- [ ] Apply `SCALAR_FACTOR` (×1000 where `thousands`); keep `STATUS` as `quality_flag`; **do not impute** suppressed values
- [ ] Filter to the agreed analysis window
- [ ] `cihi_mh_services`: unpivot chart-config rows → `indicator | breakdown | group | value | ci_low | ci_high`
- [ ] `cihi_children_youth`: split each `95% CI` string into `ci_low` / `ci_high`; tidy long
- [ ] `mhacs_2022_pumf`: replace non-response codes (6/7/8/9, 96, 996, 99.6 …) with NaN for the selected variables only
- [ ] Save each cleaned dataset to `data/processed/` and note row counts


In [ ]:
# StatCan long-format CSVs file
# Columns include:
# REF_DATE, GEO, Age group, Sex, Gender, Indicators, Characteristics,
# Statistics, UOM, VALUE, STATUS, SYMBOL, TERMINATED, DECIMALS
# Cleaning tasks:
# Strip whitespace
# Standardize column names
# Convert REF_DATE to datetime or period
# Convert VALUE to numeric
# Remove suppressed rows (VALUE missing but STATUS = “suppressed”)
# Drop useless metadata columns
# Normalize GEO names
# Normalize Indicators names
# Remove rows where VALUE is NA & STATUS is NA (true blanks)

def clean_statcan_long(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()  # create a copy of the input DataFrame so the original isn’t modified.
                    # this will avoid the effect on the upstream code and safer and more reusable.

    # strip whitespace 
    df.columns = df.columns.str.strip()
    for col in df.select_dtypes(include="object"): # use loop for every column with object dtype (usually strings)
                                                                                                 
        df[col] = df[col].astype(str).str.strip()  # casts values to string and removes spaces from cell
                                                  

    # standardize column names, used the snake_case style
    rename_map = {
        "Age group": "age_group",
        "Sex": "sex",
        "Gender": "gender",
        "GEO": "geo",
        "Indicators": "indicator",
        "Characteristics": "characteristic",
        "Statistics": "statistic",
        "UOM": "uom",
        "REF_DATE": "ref_date",
        "VALUE": "value",
        "DGUID" : "dguid",
        "UOM_ID" : "uom_id",
        "SCALAR_FACTOR" : "scalar_factor",
        "SCALAR_ID":"scalar_id",
        "VECTOR":"vector",
        "COORDINATE":"coordinate",
        "STATUS":"status"

    }
    df.rename(columns=rename_map, inplace=True)

    # convert ref_date to datetime object, used the try-except for stopping the crash if date does not exist
    df["ref_date"] = df["ref_date"].astype(str)
    try:
        df["ref_date"] = pd.to_datetime(df["ref_date"])
    except:
        pass 

    # Convert the value to int/flt  
    if "value" in df.columns:
        df["value"] = pd.to_numeric(df["value"], errors="coerce")

    # remove suppressed rows, will only keep valid data good for analyzation
    if "STATUS" in df.columns:
        df = df[~df["STATUS"].str.contains("suppressed", na=False)]
        #df = df.drop(columns=["status"])

    # drop useless metadata 
    drop_cols = ["SYMBOL", "TERMINATED", "DECIMALS"]
    df = df.drop(columns=[c for c in drop_cols if c in df.columns])

    # normalize GEO - Replaces " / " with "/" in geography names- (eg: "Newfoundland / Labrador" vs "Newfoundland/Labrador")
    if "geo" in df.columns:
        df["geo"] = df["geo"].str.replace(" / ", "/", regex=False)

    # normalize indicator names - remove double spaces to single space
    if "indicator" in df.columns:
        df["indicator"] = df["indicator"].str.replace("  ", " ", regex=False)

    return df


In [16]:
# CIHI vizconfig CSV
# Need to be converted from pivot to unpivot 

def clean_cihi_vizconfig(df: pd.DataFrame) -> pd.DataFrame: # define func that takes a pandas DataFrame and returns a cleaned DataFrame
    df = df.copy()

    # Takes the x_axis_values column, which contains strings like "2018,2019,2020", and 
    # converts each cell into a list: ["2018", "2019", "2020"].
    # Does the same for y_axis_values, e.g. "12.3,14.1,15.0" → ["12.3", "14.1", "15.0"]
    df["x_axis_values"] = df["x_axis_values"].astype(str).str.split(",")
    df["y_axis_values"] = df["y_axis_values"].astype(str).str.split(",")

    # unpacking chart into multiple data points

    tidy_rows = []  # Initializes an empty list that will store dictionaries, each representing one cleaned row


    for _, row in df.iterrows():  # df.iterrows() iterates over each row in the DataFrame 
        indicator = row["indicator"] # indicator gets the indicator name (e.g.“Depression prevalence”)
        xs = row["x_axis_values"] # xs gets the list of x-axis values (e.g.years or categories)
        ys = row["y_axis_values"] # ys gets the list of y-axis values (e.g.percentages or rates)

        # ensure equal length
        n = min(len(xs), len(ys))

    # looping for removing spaces from the x-axis and y-axis value
        for i in range(n):
            x = xs[i].strip()
            y = ys[i].strip()

            value = pd.to_numeric(y, errors="coerce")  # pd.to_numeric(y, errors="coerce") tries to convert y to a number:
                                                       # eg: if y is like "12.3", it becomes 12.3.
                                                       # eg: if y is "F" or ".." or "Suppressed", it becomes NaN instead of crashing.
    
    # Creates a dictionary representing one cleaned data point
            tidy_rows.append({
                "indicator": indicator, 
                "breakdown": row.get("breakdown", None), 
                "group": x,
                "value": value,
                "ci_low": pd.to_numeric(row.get("ci_low", None), errors="coerce"),
                "ci_high": pd.to_numeric(row.get("ci_high", None), errors="coerce"),
            })

    # Convert the list of dictionary to DataFrame
    tidy = pd.DataFrame(tidy_rows)

    # Now remove NaN rows 
    # the Nan values were came from values which were not numeric instead were "F", "...", "Suppressed" etc.
    
    tidy = tidy.dropna(subset=["value"])

    return tidy


In [ ]:
#CIHI Children/Youth Excel
# There are two sheets - Table8DATA_to hide , Table13DATA_to hide
# Cleaning tasks:
# Remove title row
# Standardize headers
# Convert fiscal years to numeric
# convert from wide spreading → long spreading
# Normalize diagnosis names

def clean_cihi_children_youth(raw_excel: dict) -> pd.DataFrame: # Defines a function that takes raw_excel, a dictionary where:
                                                                # keys = sheet names (e.g. "Table8DATA_to hide")
                                                                # values = DataFrames parsed from those sheets

    frames = [] # Initializes frames as an empty list that will store cleaned DataFrames from each sheet,
                # this file has multiple sheets, here it will combine all the sheets

    for sheet, df in raw_excel.items(): # Iterates over each (sheet_name, DataFrame) pair in raw_excel
                                        # Each sheet contains similar structured data (e.g.ED visits, hospitalizations)
                                        # You’ll apply the same cleaning logic to each sheet
        df = df.copy()      # Creating a copy so it shouldn't affect the  original

        # standardize headers - removes spaces from column names, replaces newline characters in headers with spaces
        df.columns = df.columns.str.strip().str.replace("\n", " ")

        # Identify years columns vs ID columns
        year_cols = [c for c in df.columns if "20" in c] # year_cols: selects columns whose names contain "20" (e.g."2018-2019", "2020-2021")
        id_cols = [c for c in df.columns if c not in year_cols] # id_cols: all other columns these are identifier columns (e.g. diagnosis, age group, sex)
                                                                # the file has multiple sheets as you keep one column for year one column for each ID,
                                                                # it will be easy to analyze trends

        long_df = df.melt(id_vars=id_cols, value_vars=year_cols,   # we're transforming dataframe. eg: Dataframe was before x-axis values: Diagnosis, Sex	2018-2019	2019-2020
                          var_name="fiscal_year", value_name="value")                                                        # y-axis values: Anxiety	F	12.3	13.1 (before)
                                                                   # (After transf.) x-axis values: Diagnosis, Sex, fiscal_year , Value (new col for yeears and values )
                                                                   # We are transforming the sheet from horizontally spreading to vertical spreading, for understanding the trends
         

        # remove extra spaces from fiscal year
        long_df["fiscal_year"] = long_df["fiscal_year"].str.replace(" ", "")

        # Collect the cleaned long DataFrame in frame list
        frames.append(long_df)
     
    return pd.concat(frames, ignore_index=True) # Concatenate all cleaned sheets into one DataFrame, this will stack all dataframes vertically in frames list


In [18]:
def clean_mhacs_pumf(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # as mention in the document above there are code missing , handling it with NaN is necessary so analysis treat it as missing value not a value to analyze.
    missing_codes = {6,7,8,9,96,996,999,99.6}
    for col in df.columns:
        df[col] = df[col].replace(list(missing_codes), np.nan)

    # ensure weight numeric
    if "WTS_M" in df.columns:
        df["WTS_M"] = pd.to_numeric(df["WTS_M"], errors="coerce")

    return df


In [19]:
CLEANED = ROOT / "data" / "processed" / "02_cleaned" # Builds a path object pointing to the folder where you want to store all cleaned datasets
CLEANED.mkdir(parents=True, exist_ok=True) # Creates that folder if it doesn’t exist

cleaned = {} # Creates an empty dictionary that will store cleaned DataFrames keyed by dataset name, e.g.: 
             # cleaned["perceived_mh_annual"], cleaned["cihi_mh_services"], cleaned["mhacs_2022_pumf"]

for key, spec in DATASETS.items():
    print(f"\n=== Cleaning {key} ===")

    if spec["kind"] == "statcan_long": # Calls your clean_statcan_long function on the raw DataFrame:
        df = clean_statcan_long(raw[key])   # raw[key] is the raw version loaded earlier
        cleaned[key] = df                     # Stores the cleaned DataFrame in the cleaned dict
        df.to_csv(CLEANED / f"{key}.csv", index=False) # Writes the cleaned DataFrame to disk as a CSV: data/processed/02_cleaned

    elif spec["kind"] == "cihi_vizconfig":  # Same process for rest all
        df = clean_cihi_vizconfig(raw[key])
        cleaned[key] = df
        df.to_csv(CLEANED / f"{key}.csv", index=False)

    elif spec["kind"] == "excel_multitable":
        df = clean_cihi_children_youth(raw_excel)
        cleaned[key] = df
        df.to_csv(CLEANED / f"{key}.csv", index=False)

    elif spec["kind"] == "microdata":
        df = clean_mhacs_pumf(raw[key])
        cleaned[key] = df
        df.to_csv(CLEANED / f"{key}.csv", index=False)

print("\nALL DATASETS CLEANED ✔") # Prints final msg



=== Cleaning perceived_mh_annual ===

=== Cleaning suicidal_thoughts ===

=== Cleaning stress_coping ===

=== Cleaning perceived_health_quarterly ===

=== Cleaning cchs_mh_disorders ===

=== Cleaning cihi_mh_services ===

=== Cleaning cihi_children_youth ===

=== Cleaning mhacs_2022_pumf ===

ALL DATASETS CLEANED ✔
